In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
import pandas as pd
import os
from scipy.signal import butter, filtfilt, medfilt, hilbert
from scipy.fft import fft, ifft
from scipy.signal import iirnotch
!pip install PyWavelets
import pywt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#radio (R) , vangular (w)  , distancia cntro-objeto (D) , velocidad sonido (v)
#frecuancia emita (f) , frecuencia percibida(fp) , tiempo (t)
def efecto_doppler(w,f,t):
  R = 0.14606
  D = R +0.00413
  v = 343
  epsilon = R/D
  fp = f* ( 1- ( (R*w/v) * np.sin(w*t) * (1/( 1+ epsilon**2 - 2*epsilon*np.cos(w*t) )**1/2) )  )
  return fp

In [ ]:
# Función para aplicar un filtro pasa banda
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

# Función para aplicar un filtro de promedio móvil
def moving_average(signal, window_size):
    return np.convolve(signal, np.ones(window_size)/window_size, mode='same')

# Función para eliminar interferencias específicas en el dominio de Fourier
def remove_frequency_interference(signal, fs, freqs_to_remove, bandwidth=5):
    spectrum = fft(signal)
    frequencies = np.fft.fftfreq(len(signal), d=1/fs)
    for f in freqs_to_remove: # Eliminar frecuencias indeseadas
        spectrum[(frequencies > f - bandwidth) & (frequencies < f + bandwidth)] = 0
    return np.real(ifft(spectrum))

def wavelet_denoising(signal, wavelet="db8", level=4):
    """ Elimina ruido con transformada wavelet """
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    coeffs[1:] = [pywt.threshold(c, np.std(c) * 0.5, mode="soft") for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)



In [ ]:
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def moving_average(signal, window_size=10):
    return np.convolve(signal, np.ones(window_size)/window_size, mode='same')

def wavelet_denoising(signal):
    # Placeholder for wavelet denoising function
    return signal  # Implement proper wavelet denoising if needed

def remove_frequency_interference(signal, fs, freqs_to_remove):
    # Placeholder for frequency interference removal
    return signal  # Implement proper notch filter if needed

def remove_outliers(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.clip(data, lower_bound, upper_bound)

def load_audio(file_path):
    fs, signal = wavfile.read(file_path)

    if signal.ndim > 1:
        signal = signal.mean(axis=1)  # Convert to mono

    lowcut, highcut = 300, 20000
    filtered_signal = butter_bandpass_filter(signal, lowcut, highcut, fs)
    filtered_signal = medfilt(filtered_signal, kernel_size=5)
    filtered_signal = moving_average(filtered_signal, window_size=10)
    filtered_signal = wavelet_denoising(filtered_signal)
    filtered_signal = remove_frequency_interference(filtered_signal, fs, freqs_to_remove=[50, 60])

    analytic_signal = hilbert(filtered_signal)
    instantaneous_phase = np.unwrap(np.angle(analytic_signal))
    instantaneous_frequency = np.diff(instantaneous_phase) * fs / (2.0 * np.pi)
    instantaneous_frequency = np.clip(instantaneous_frequency, 0, 2 * fs)

    instantaneous_frequency = remove_outliers(instantaneous_frequency)
    time = np.arange(len(instantaneous_frequency)) / fs

    return time, instantaneous_frequency

In [ ]:
# audio1= "/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-01-440hz-16rev.wav"
# audio2="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-02-550hz-16rev (mp3cut.net).wav"
# audio3="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-03-440hz-33rev (mp3cut.net).wav"
# audio4="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-04-550hz-33rev (mp3cut.net).wav"
# audio5= "/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-05fuentemoviendo-440hz-16rev (mp3cut.net).wav"
# audio6="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-06fuentemoviendo-550hz-16rev (mp3cut.net).wav"
# audio7="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-07fuentemoviendo-550hz-78rev (mp3cut.net).wav"
# audio8="/content/drive/MyDrive/Física Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-08fuentemoviendo-440hz-78rev (mp3cut.net).wav"

# w1 , w2 , w3 , w4 , w5 , w6 , w7 , w8 = 1.67552 , 1.67552 , 3.508112 , 3.508112 , 1.67552 , 1.67552 , 8.16814 , 8.16814
# f1 , f2 , f3 , f4 , f5 , f6 , f7 , f8 = 440 ,550, 440, 550, 440, 550, 550, 440

In [ ]:
audio1= "/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-01-440hz-16rev.wav"
audio2="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-02-550hz-16rev (mp3cut.net).wav"
audio3="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-03-440hz-33rev (mp3cut.net).wav"
audio4="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-04-550hz-33rev (mp3cut.net).wav"
audio5= "/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-05fuentemoviendo-440hz-16rev (mp3cut.net).wav"
audio6="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-06fuentemoviendo-550hz-16rev (mp3cut.net).wav"
audio7="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-07fuentemoviendo-550hz-78rev (mp3cut.net).wav"
audio8="/content/drive/MyDrive/Experimental /Experimental III/Proyecto Final/audios-medidas/grabacion-proyecto-final-doppler-08fuentemoviendo-440hz-78rev (mp3cut.net).wav"

w1 , w2 , w3 , w4 , w5 , w6 , w7 , w8 = 1.67552 , 1.67552 , 3.508112 , 3.508112 , 1.67552 , 1.67552 , 8.16814 , 8.16814
f1 , f2 , f3 , f4 , f5 , f6 , f7 , f8 = 440 ,550, 440, 550, 440, 550, 550, 440

In [ ]:
señal1 , señal2, señal3, señal4 = load_audio(audio1) , load_audio(audio2) , load_audio(audio3) , load_audio(audio4)
señal5 , señal6, señal7, señal8 = load_audio(audio5) , load_audio(audio6) , load_audio(audio7) , load_audio(audio8)

t = np.linspace(0, 20, 10000)
teoria1 , teoria2 , teoria3, teoria4 = efecto_doppler(w1,f1,t) , efecto_doppler(w2,f2,t) , efecto_doppler(w3,f3,t) , efecto_doppler(w4,f4,t)
teoria5 , teoria6 , teoria7, teoria8 = efecto_doppler(w5,f5,t) , efecto_doppler(w6,f6,t) , efecto_doppler(w7,f7,t) , efecto_doppler(w8,f8,t)

In [ ]:
# Definir los valores de frecuencia para centrar los ejes
f_values = [440, 550, 440, 550, 440, 550, 550, 440]

# Definir los valores de velocidad angular
w_values = [1.67552, 1.67552, 3.508112, 3.508112, 1.67552, 1.67552, 8.16814, 8.16814]

# Configuración de la figura y subgráficos
fig, axes = plt.subplots(8, 1, figsize=(15, 30))

# Iterar sobre las señales experimentales y teóricas
for i, (señal, teoria, f_centro, w_valor) in enumerate(zip(
    [señal1, señal2, señal3, señal4, señal5, señal6, señal7, señal8],
    [teoria1, teoria2, teoria3, teoria4, teoria5, teoria6, teoria7, teoria8],
    f_values,
    w_values
)):
    ax = axes[i]

    # Graficar la señal experimental
    exp_line, = ax.plot(señal[0], señal[1], color="blue", label="Señal Experimental", linewidth=0.5)

    # Segundo eje Y para el modelo teórico
    ax2 = ax.twinx()
    teorica_line, = ax2.plot(t, teoria, color="red", linestyle="dashed", label="Modelo Teórico", linewidth=1.5)

    # Añadir línea horizontal negra en la frecuencia central
    linea_central = ax.axhline(y=f_centro, color='black', linestyle='-', linewidth=1.5, label=f'Frecuencia Original: {f_centro} Hz')
    ax2.axhline(y=f_centro, color='black', linestyle='-', linewidth=1.5)

    # Configuración del eje de la señal experimental
    ax.set_ylabel("Frecuencia Exp. (Hz)", fontsize=12, color="blue")
    ax.set_xlim(0, abs(señal[0][-1] - señal[0][0]))

    # Calcular los límites del eje Y basados en el valor central
    y_min_exp = min(señal[1])
    y_max_exp = max(señal[1])
    y_centro_exp = (y_min_exp + y_max_exp) / 2
    y_rango_exp = y_max_exp - y_min_exp

    # Establecer límites del eje Y extendiendo el rango pero manteniendo el centro
    ax.set_ylim(f_centro - y_rango_exp*2/2, f_centro + y_rango_exp*2/2)
    ax.tick_params(axis='y', labelcolor="blue")

    # Configuración del eje del modelo teórico
    ax2.set_ylabel("Frecuencia Teórica (Hz)", fontsize=12, color="red")

    # Calcular los límites del eje Y para el segundo eje
    y_teoria_range = ax2.get_ylim()[1] - ax2.get_ylim()[0]
    ax2.set_ylim(f_centro - y_teoria_range/2, f_centro + y_teoria_range/2)
    ax2.tick_params(axis='y', labelcolor="red")

    # Añadir título específico por cada gráfico con frecuencia y velocidad angular
    ax.set_title(f"Comparación Experimental vs Teórica - Caso {i+1}\nFrecuencia: {f_centro} Hz, Velocidad Angular: {round(w_valor, 1)} rad/s",
                 fontsize=14, fontweight='bold')

    # Añadir líneas de referencia de cuadrícula
    ax.grid(True, linestyle="--", linewidth=0.6)

    # Añadir leyendas
    lines = [exp_line, teorica_line, linea_central]
    labels = ["Señal Experimental", "Modelo Teórico", f'Frecuencia Original']
    ax.legend(lines, labels, loc='upper left')

# Configuración general de la figura
fig.suptitle("Análisis Comparativo de Frecuencias: Señal Experimental vs Modelo Teórico", fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()

# Mostrar la figura
plt.show()

In [ ]:
# Definir los valores de frecuencia para centrar los ejes
f_values = [440, 550, 440, 550, 440, 550, 550, 440]

# Definir los valores de velocidad angular
w_values = [1.67552, 1.67552, 3.508112, 3.508112, 1.67552, 1.67552, 8.16814, 8.16814]

señales = [señal1, señal2, señal3, señal4, señal5, señal6, señal7, señal8]
teorias = [teoria1, teoria2, teoria3, teoria4, teoria5, teoria6, teoria7, teoria8]

# Función para generar la figura
def plot_signals(indices, title):
    fig, axes = plt.subplots(len(indices), 1, figsize=(15, 15), sharex=True)

    for j, i in enumerate(indices):
        ax = axes[j]

        # Graficar la señal experimental
        exp_line, = ax.plot(señales[i][0], señales[i][1], color="#9ec3db", label="Señal experimental", linewidth=0.5)

        # Segundo eje Y para el modelo teórico
        ax2 = ax.twinx()
        teorica_line, = ax2.plot(t, teorias[i], color="#73246a", linestyle="dashed", label="Modelo teórico", linewidth=1.5)

        # Añadir línea horizontal negra en la frecuencia central
        linea_central = ax.axhline(y=f_values[i], color='black', linestyle='-', linewidth=1.5, label=f'Frecuencia original: {f_values[i]} Hz')
        ax2.axhline(y=f_values[i], color='black', linestyle='-', linewidth=1.5)

        # Configuración del eje de la señal experimental
        ax.set_ylabel("Frecuencia exp. (Hz)", fontsize=12, color="#9ec3db")
        ax.set_xlim(0, abs(señales[i][0][-1] - señales[i][0][0]))

        # Mantener los mismos límites del eje Y
        ax.set_ylim(f_values[i] - y_rango_exp*2/2, f_values[i] + y_rango_exp*2/2)
        ax.tick_params(axis='y', labelcolor="#9ec3db")

        # Configuración del eje del modelo teórico
        ax2.set_ylabel("Frecuencia teórica (Hz)", fontsize=12, color="#73246a")
        ax2.set_ylim(f_values[i] - y_teoria_range/2, f_values[i] + y_teoria_range/2)
        ax2.tick_params(axis='y', labelcolor="#73246a")

        # Añadir título específico por cada gráfico
        ax.set_title(f"Comparación experimental vs teórica - Caso {i+1}\nFrecuencia: {f_values[i]} Hz, Velocidad Angular: {round(w_values[i], 1)} rad/s",
                     fontsize=14, fontweight='bold')

        # Añadir líneas de referencia de cuadrícula
        ax.grid(True, linestyle="--", linewidth=0.6)

        # Añadir leyendas
        lines = [exp_line, teorica_line, linea_central]
        labels = ["Señal experimental", "Modelo teórico", f'Frecuencia original']
        ax.legend(lines, labels, loc='upper left')

    # Agregar etiqueta global del eje X
    fig.text(0.5, 0.001, "Tiempo (segundos)", ha='center', fontsize=14)

    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])  # Adjusting bottom, left, right, and top limits
    fig.savefig(f"{title}.png")
    plt.show()

In [ ]:
plot_signals(range(4), "Análisis comparativo de frecuencias: Señal experimental vs modelo teórico - Receptor en movimiento circularazules")

In [ ]:
plot_signals(range(4, 8), "Análisis comparativo de frecuencias: Señal experimental vs modelo teórico - Fuente en movimiento circularazules")